In [8]:
from pathlib import Path
import pandas as pd
import re
import numpy as np
import rasterio
from rasterio.warp import transform_bounds
from rasterio.windows import from_bounds

In [ ]:
# set paths and bounding box
path_GLWD = r'...\Global_Lakes_and_Wetlands_Database_(GLWD)\GLWD_v2_0_area_by_class_pct'
path_GLWD_legend = '...\data\Global_Lakes_and_Wetlands_Database_(GLWD)\GLWD_Legend_v2_0.csv'
bb_berlin = [13.0884, 52.3415, 13.7611, 52.6697]  # [west, south, east, north]

In [ ]:
p = Path(path_GLWD)
if not p.exists():
    raise FileNotFoundError(f"Directory not found: {p}")

geotiff_files = sorted(str(fp) for fp in p.rglob("*") if fp.suffix.lower() in (".tif", ".tiff"))
print(f"Found {len(geotiff_files)} GeoTIFF files")
geotiff_files[:20]  # preview first 20 paths

In [11]:
legend_fp = Path(path_GLWD_legend)
if not legend_fp.exists():
    raise FileNotFoundError(f"Legend file not found: {legend_fp}")

df_legend = pd.read_csv(legend_fp)
print("Legend shape:", df_legend.shape)
df_legend.head()

Legend shape: (34, 2)


,GLWD_ID,Class_name
0,0,Dryland (non-wetland)
1,1,Freshwater lake
2,2,Saline lake
3,3,Reservoir
4,4,Large river


In [12]:
west, south, east, north = bb_berlin

stats = []
for fp in geotiff_files:
    m = re.search(r'class_(\d+)_', fp)
    if not m:
        continue
    glwd_id = int(m.group(1))
    with rasterio.open(fp) as ds:
        # transform bbox to dataset CRS (if defined)
        try:
            dst_bounds = transform_bounds('EPSG:4326', ds.crs, west, south, east, north, densify_pts=21)
        except Exception:
            dst_bounds = (west, south, east, north)

        win = from_bounds(*dst_bounds, transform=ds.transform)
        arr = ds.read(1, window=win, boundless=True, fill_value=ds.nodata if ds.nodata is not None else np.nan)
        if ds.nodata is not None:
            arr = np.where(arr == ds.nodata, np.nan, arr)
        vals = arr.astype(float).ravel()
        vals = vals[~np.isnan(vals)]

        if vals.size == 0:
            mean = std = median = np.nan
            count = 0
        else:
            mean = float(np.nanmean(vals))
            std = float(np.nanstd(vals))
            median = float(np.nanmedian(vals))
            count = int(vals.size)

    stats.append({'GLWD_ID': glwd_id, 'mean': mean, 'std': std, 'median': median, 'count': count})

df_stats = pd.DataFrame(stats).sort_values('GLWD_ID').reset_index(drop=True)
df_stats = df_legend.merge(df_stats, on='GLWD_ID', how='left')
df_stats

,GLWD_ID,Class_name,mean,std,median,count
0,0,Dryland (non-wetland),85.664125,27.884263,100.0,12719
1,1,Freshwater lake,2.682365,13.841574,0.0,12719
2,2,Saline lake,0.000000,0.000000,0.0,12719
3,3,Reservoir,0.000000,0.000000,0.0,12719
4,4,Large river,0.416149,3.303648,0.0,12719
5,5,Large estuarine river,0.000000,0.000000,0.0,12719
6,6,Other permanent waterbody,0.145845,1.506869,0.0,12719
7,7,Small streams,0.052441,0.470200,0.0,12719
8,8,"Lacustrine, forested",0.359148,2.089653,0.0,12719
9,9,"Lacustrine, non-forested",0.107949,0.793122,0.0,12719


In [ ]:
# get dataframe with mean, std, and median value for each class defined in the legend from all geotiff_files for the bounding box of Berlin